# Thermal-Contrast — обучение, eval и маска по `.mat`

Клип кадров из термо-видео → **contrast maps** (max−min, max−t₀, σ) → static U-Net → **маска сегментации**.

| Раздел | Что делает |
|--------|------------|
| 1 | Превью collapse-пайплайна |
| 2 | Обучение + live-графики |
| 3 | Eval на test split (метрики + визуализации) |
| 4 | **Inference**: произвольный `.mat` → маска |

Перед запуском: кэш `artifacts/cache/` (см. ячейку проверки).


In [ ]:
# === параметры ===
PRESET = "combo"              # minimal | delta | combo | pca | full
EPOCHS = 25
BATCH_SIZE = 8
TEST_EVERY = 4
NUM_WORKERS = 0
LR = 3e-4
WEIGHT_DECAY = 1e-4
POS_WEIGHT = 10.0
COMPILE = False
DEVICE = "auto"
RESUME = None                 # None | "best" | "last" | path/to.ckpt
PREVIEW_EVERY = 2             # превью pred каждые N эпох
YAML = None                   # None → models/Thermal-Contrast/dataset_contrast.yaml

# --- inference на .mat ---
MAT_PATH = "data/sample10.mat"  # путь к вашему .mat
TIME_AXIS = None                # None = авто (sample* → 2)
FRAME_RANGE = None              # None = как в dataset_contrast.yaml
MASK_OUT = "runs/contrast_pred/sample10_mask.png"
THRESHOLD = 0.5


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import clear_output, display
from PIL import Image

CHANNEL_TITLES = {
    "maxmin": "max − min",
    "maxfirst": "max − t₀",
    "minfirst": "min − t₀",
    "lastfirst": "last − t₀",
    "std": "σ(t)",
    "mean": "⟨t⟩",
    "pca1": "PCA₁(t)",
}


def find_project_root() -> Path:
    start = Path.cwd().resolve()
    candidates = [start]
    if start.name == "notebooks":
        candidates.append(start.parent)
    for base in candidates:
        for p in [base, *base.parents]:
            p = p.resolve()
            if (p / "data").is_dir() and (p / "irt_data").is_dir():
                return p
    raise FileNotFoundError(f"корень репо не найден, cwd={start}")


ROOT = find_project_root()
TC = ROOT / "models" / "Thermal-Contrast"
SEG = ROOT / "models"
if str(TC) not in sys.path:
    sys.path.insert(0, str(TC))
for p in (SEG, ROOT):
    sp = str(p)
    if sp not in sys.path:
        sys.path.append(sp)

for _name in ("inference", "data", "train", "features", "model", "irt_cfg"):
    sys.modules.pop(_name, None)

from common.device import get_device
from common.mps_train import load_model_weights, setup_mps_env, suggest_num_workers
from common.metrics import dice_score, iou_score
from data import build_contrast_loaders
from features import PRESETS, collapse_temporal, normalize_contrast
from inference import evaluate_loader, load_gt_mask_for_mat, predict_mat, resolve_gt_mask_path
from irt_cfg import load_cfg
from model import UNetModel
from train import train_one

setup_mps_env()
%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

device = get_device(DEVICE)
yaml_path = TC / "dataset_contrast.yaml" if YAML is None else Path(YAML)
if not yaml_path.is_absolute():
    yaml_path = (ROOT / yaml_path).resolve()
eval_cfg = load_cfg(yaml_path, train=False)
nw = suggest_num_workers(device) if NUM_WORKERS is None else NUM_WORKERS

print(f"ROOT={ROOT.name}  device={device}  preset={PRESET}")
print(f"channels={PRESETS[PRESET]}")
print(f"yaml={yaml_path.relative_to(ROOT)}")


In [ ]:
cache_index = ROOT / "artifacts" / "cache" / "index.json"
if not cache_index.exists():
    print("⚠ Кэш не найден. Сначала выполните из корня репо:")
    print("  python scripts/build_irt_cache.py --yaml models/Thermal-Contrast/dataset_contrast.yaml")
else:
    print(f"✓ cache OK: {cache_index}")

train_loader, test_loader, train_ds, test_ds = build_contrast_loaders(
    yaml_path,
    preset=PRESET,
    test_every=TEST_EVERY,
    batch_size=BATCH_SIZE,
    num_workers=nw,
)
print(
    f"train {len(train_ds.video_ids)} vid / {len(train_ds)} samples\n"
    f"test  {len(test_ds.video_ids)} vid / {len(test_ds)} samples"
)


def ckpt_path(preset: str = PRESET, which: str = "best") -> Path:
    return TC / f"model_contrast_{preset}_{which}.tar"


def load_trained_model(preset: str = PRESET, which: str = "best") -> UNetModel:
    path = ckpt_path(preset, which)
    if not path.exists():
        raise FileNotFoundError(f"checkpoint not found: {path}")
    model = UNetModel(in_channels=len(PRESETS[preset])).to(device)
    load_model_weights(model, path)
    model.eval()
    print(f"loaded {path.name}")
    return model


## 1. Как клип сворачивается в contrast maps

In [ ]:
def show_collapse_pipeline(ds, idx: int = 0, *, max_frames: int = 8) -> None:
    base = ds.base
    s = base[idx]
    raw = s["image"].numpy()
    T = raw.shape[0]
    pick = np.linspace(0, T - 1, num=min(max_frames, T), dtype=int)

    feat_np = normalize_contrast(collapse_temporal(raw[:, 0], ds.channels))
    mask_np = ds[idx][1].squeeze().numpy()
    n_ch = len(ds.channels)

    fig1, ax1 = plt.subplots(figsize=(14, 2.2))
    strip = np.hstack([raw[t, 0] for t in pick])
    ax1.imshow(strip, cmap="inferno", aspect="auto")
    ax1.set_title(f"клип T={T}, video={s['video_id']!r}")
    ax1.set_yticks([])
    ax1.set_xticks([])
    fig1.tight_layout()
    plt.show()

    fig2, ax2 = plt.subplots(1, n_ch + 1, figsize=(2.6 * (n_ch + 1), 2.8))
    if n_ch + 1 == 1:
        ax2 = [ax2]
    for c, name in enumerate(ds.channels):
        ax2[c].imshow(feat_np[c], cmap="magma")
        ax2[c].set_title(CHANNEL_TITLES.get(name, name))
        ax2[c].axis("off")
    ax2[-1].imshow(mask_np, cmap="gray", vmin=0, vmax=1)
    ax2[-1].set_title(f"GT pos={mask_np.mean():.3f}")
    ax2[-1].axis("off")
    fig2.suptitle("клип → contrast channels → GT", fontsize=11)
    fig2.tight_layout()
    plt.show()


show_collapse_pipeline(train_ds, 0)


## 2. Обучение с live-графиками

In [ ]:
def plot_training_dashboard(history, *, epoch, end_epoch, best_iou, best_epoch, title=""):
    if not history:
        return
    ep = [float(r["epoch"]) for r in history]
    fig, axes = plt.subplots(2, 2, figsize=(11, 7))
    fig.suptitle(title or f"Thermal-Contrast [{PRESET}]  epoch {epoch}/{end_epoch}", fontsize=12)

    specs = [
        ("train_loss", "test_loss", "Loss"),
        ("train_iou", "test_iou", "IoU"),
        ("train_dice", "test_dice", "Dice"),
    ]
    for ax, (tk, vk, ylab) in zip(axes.flat[:3], specs):
        tr = [float(r[tk]) for r in history]
        te = [float(r[vk]) for r in history]
        ax.plot(ep, tr, "o-", label="train", ms=4, lw=1.5)
        ax.plot(ep, te, "s-", label="test", ms=4, lw=1.5)
        if best_epoch in ep:
            j = ep.index(float(best_epoch))
            ax.axvline(best_epoch, color="green", ls="--", alpha=0.4, label=f"best@{best_epoch}")
        ax.set_title(ylab)
        ax.set_xlabel("epoch")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    ax = axes[1, 1]
    ax.axis("off")
    last = history[-1]
    lines = [
        f"epoch {epoch} / {end_epoch}",
        f"train loss {last['train_loss']:.4f}  test {last['test_loss']:.4f}",
        f"train IoU  {last['train_iou']:.4f}  test {last['test_iou']:.4f}",
        f"train Dice {last['train_dice']:.4f}  test {last['test_dice']:.4f}",
        f"best test IoU {best_iou:.4f} @ {best_epoch}",
        f"preset={PRESET}  ch={train_ds.channels}",
    ]
    ax.text(0.05, 0.95, "\n".join(lines), va="top", family="monospace", fontsize=10,
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.3))
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def preview_prediction(model, ds, device, *, idx=0, title=""):
    model.eval()
    feat, mask = ds[idx]
    with torch.no_grad():
        prob = torch.sigmoid(model(feat.unsqueeze(0).to(device))).squeeze().cpu().numpy()
    pred = (prob > 0.5).astype(np.float32)
    gt = mask.squeeze().numpy()
    n = feat.shape[0]
    from common.metrics import dice_score, iou_score
    d = dice_score(pred, gt)
    iou = iou_score(pred, gt)

    fig, axes = plt.subplots(2, max(n, 3), figsize=(2.4 * max(n, 3), 5))
    for c in range(n):
        axes[0, c].imshow(feat[c].numpy(), cmap="magma")
        axes[0, c].set_title(ds.channels[c], fontsize=9)
        axes[0, c].axis("off")
    for c in range(n, axes.shape[1]):
        axes[0, c].axis("off")

    axes[1, 0].imshow(gt, cmap="gray", vmin=0, vmax=1)
    axes[1, 0].set_title("GT")
    axes[1, 0].axis("off")
    axes[1, 1].imshow(prob, cmap="magma", vmin=0, vmax=1)
    axes[1, 1].set_title("prob")
    axes[1, 1].axis("off")
    axes[1, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, 2].set_title(f"pred Dice={d:.3f}")
    axes[1, 2].axis("off")
    mid = min(1, n - 1)
    if n > 1:
        axes[1, mid].imshow(feat[mid].numpy(), cmap="magma")
        axes[1, mid].contour(pred, colors="lime", levels=[0.5], linewidths=1)
        axes[1, mid].set_title("overlay")
        axes[1, mid].axis("off")
    for c in range(3, axes.shape[1]):
        axes[1, c].axis("off")
    if title:
        fig.suptitle(f"{title}  IoU={iou:.3f}", fontsize=10)
    fig.tight_layout()
    display(fig)
    plt.close(fig)


def _on_epoch_end(tracker, row, epoch, end_epoch, best_iou, best_epoch, opt, model):
    clear_output(wait=True)
    print(tracker.format_line(row, end_epoch))
    print(f"lr={opt.param_groups[0]['lr']:.2e}")
    plot_training_dashboard(
        tracker.history,
        epoch=epoch,
        end_epoch=end_epoch,
        best_iou=best_iou,
        best_epoch=best_epoch,
    )
    if PREVIEW_EVERY and epoch % PREVIEW_EVERY == 0:
        preview_prediction(model, test_ds, device, idx=0, title=f"test[0] @ epoch {epoch}")


In [ ]:
tracker, model, best_iou, best_epoch = train_one(
    PRESET,
    yaml_path,
    EPOCHS,
    device,
    test_every=TEST_EVERY,
    batch_size=BATCH_SIZE,
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    pos_weight=POS_WEIGHT,
    compile_model=COMPILE,
    num_workers=nw,
    resume=RESUME,
    on_epoch_end=_on_epoch_end,
)

ckpt_best = ckpt_path(PRESET, "best")
print(f"\n=== done: best test IoU {best_iou:.4f} @ epoch {best_epoch} ===")
print(f"checkpoint: {ckpt_best}")
plot_training_dashboard(
    tracker.history,
    epoch=int(tracker.history[-1]["epoch"]),
    end_epoch=int(tracker.history[-1]["epoch"]),
    best_iou=best_iou,
    best_epoch=best_epoch,
    title=f"FINAL — {PRESET}",
)


## 3. Eval на test split

Запускай **после обучения** или **отдельно**: модель подгрузится с диска автоматически.


In [ ]:
try:
    model  # noqa: B018
except NameError:
    model = load_trained_model(PRESET, "best")
elif ckpt_path(PRESET, "best").exists():
    load_model_weights(model, ckpt_path(PRESET, "best"))
    print(f"reloaded {ckpt_path(PRESET, 'best').name}")

test_loss, test_metrics, per_sample = evaluate_loader(model, test_loader, device)
print(f"test loss={test_loss:.4f}  IoU={test_metrics['iou']:.4f}  Dice={test_metrics['dice']:.4f}")

df = pd.DataFrame(per_sample).sort_values("iou", ascending=False)
display(df.style.format({"dice": "{:.3f}", "iou": "{:.3f}", "gt_pos": "{:.3f}", "pred_pos": "{:.3f}"}))

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(df["video_id"], df["iou"], color="steelblue", alpha=0.85)
ax.set_ylabel("IoU")
ax.set_title("Per-video IoU (test split)")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
plt.show()

for i in range(len(test_ds)):
    preview_prediction(model, test_ds, device, idx=i, title=f"test[{i}] {test_ds.base[i]['video_id']}")


## 4. Маска сегментации для `.mat`

`FRAME_RANGE = None` — как при обучении.


In [ ]:
def show_mat_prediction(result: dict, *, gt: np.ndarray | None = None) -> None:
    feat = result["feat"]
    prob = result["prob"]
    pred = result["pred"]
    ch = result["channels"]
    n = feat.shape[0]

    fig, axes = plt.subplots(2, max(n, 3) + 1, figsize=(2.5 * (max(n, 3) + 1), 5))
    for c in range(n):
        axes[0, c].imshow(feat[c], cmap="magma")
        axes[0, c].set_title(CHANNEL_TITLES.get(ch[c], ch[c]), fontsize=9)
        axes[0, c].axis("off")
    axes[0, -1].imshow(prob, cmap="magma", vmin=0, vmax=1)
    axes[0, -1].set_title("prob")
    axes[0, -1].axis("off")

    col = 0
    if gt is not None:
        axes[1, col].imshow(gt, cmap="gray", vmin=0, vmax=1)
        axes[1, col].set_title("GT")
        axes[1, col].axis("off")
        col += 1
    axes[1, col].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, col].set_title("pred mask")
    axes[1, col].axis("off")
    col += 1
    axes[1, col].imshow(feat[0], cmap="magma")
    axes[1, col].contour(pred, colors="lime", levels=[0.5], linewidths=1.2)
    axes[1, col].set_title("overlay")
    axes[1, col].axis("off")

    fig.suptitle(Path(result["mat_path"]).name, fontsize=11)
    fig.tight_layout()
    plt.show()


mat_path = (ROOT / MAT_PATH).resolve()
fr = tuple(FRAME_RANGE) if FRAME_RANGE else None

try:
    model  # noqa: B018
except NameError:
    model = load_trained_model(PRESET, "best")

result = predict_mat(
    model,
    mat_path,
    eval_cfg,
    device,
    preset=PRESET,
    time_axis=TIME_AXIS,
    frame_range=fr,
    threshold=THRESHOLD,
)

stem = mat_path.stem.replace(" ", "_")
gt = load_gt_mask_for_mat(mat_path, eval_cfg, result["pred"].shape, root=ROOT)
mp = resolve_gt_mask_path(stem, eval_cfg, ROOT)
if gt is not None and mp is not None:
    d = dice_score(result["pred"], gt)
    iou = iou_score(result["pred"], gt)
    print(f"GT (aligned crop): {mp.relative_to(ROOT)}  Dice={d:.4f}  IoU={iou:.4f}")
else:
    print("GT mask не найдена (inference работает без неё)")

show_mat_prediction(result, gt=gt)

out_path = ROOT / MASK_OUT
out_path.parent.mkdir(parents=True, exist_ok=True)
Image.fromarray((result["pred"] * 255).astype(np.uint8)).save(out_path)
Image.fromarray((result["prob"] * 255).astype(np.uint8)).save(out_path.with_name(out_path.stem + "_prob.png"))
print(f"saved mask → {out_path.relative_to(ROOT)}")
